In [ ]:
# hide
import numpy as np
import pyquist as pq
from pyquist.helper import amplitude_to_db, db_to_amplitude

In [ ]:
# CLAUDE: change helper function cells (whose code has been revealed elsewhere) to # hide throughout, not # collapse
# collapse
def feedback_delay(x: np.ndarray, M: int, g: float = 1.0) -> np.ndarray:
    """A feedback delay line with attenuation, y[n] = x[n] + g * y[n - M]."""
    buffer = np.zeros(M)              # the delay line, as a circular buffer
    i = 0
    y = np.zeros(len(x))
    for n in range(len(x)):
        y[n] = x[n] + g * buffer[i]   # attenuate y[n - M] on every trip
        buffer[i] = y[n]
        i = (i + 1) % M
    return y

def wet_ratio(db: float) -> float:
    """The wet ratio w that mixes the wet signal `db` decibels above the dry signal."""
    return 1 / (1 + db_to_amplitude(-db))

In [ ]:
f_s = 44100
guitar = pq.Audio.from_file("../assets/guitar-arpeggio.wav").samples[:, 0]
guitar = np.concatenate([guitar, np.zeros(2 * f_s)])  # room for the echoes to ring out
note_interval = 0.221                         # seconds between the guitar's notes
M = round(f_s * note_interval / 2)            # echo halfway between notes: double time
wet = feedback_delay(guitar, M, g=0.75)
w = wet_ratio(6.0)                           # wet 6dB above dry
print(f"M = {M} samples, so f_s / M = {f_s / M:.1f} Hz, below 20 Hz")
pq.play(pq.Audio(guitar, f_s), normalize=True)
pq.play(pq.Audio((1 - w) * guitar + w * wet, f_s), normalize=True)